# Week-6 Assignment (~Nischal Paliwal)

---
## Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

**Driver:**
- The "brain" of the Spark application
- Runs the main program and creates the SparkContext
- Converts user code into tasks and sends them to executors
- Tracks the progress of the job

**Cluster Manager:**
- Acts like a "resource manager" (e.g., YARN)
- Allocates CPU and memory resources across the cluster

**Executor:**
- Runs on each worker node
- Actually executes the tasks assigned by the Driver
- Stores data in memory/disk for caching
- Sends results back to the Driver

**Flow:** Driver → asks Cluster Manager for resources → Cluster Manager assigns Executors → Executors run tasks

---
## Q2: How does Spark's Lazy Evaluation strategy improve performance when chain-processing large datasets?

**What is Lazy Evaluation?**
- Spark does NOT execute transformations immediately
- It only builds a plan (DAG) and waits
- Execution only starts when an **Action** is called (like `.show()`, `.count()`)

**How it improves performance:**
- Spark can **optimize the entire plan** before running (e.g., reorder filters, skip unnecessary steps)
- **Avoids reading unnecessary data** — if you filter early, Spark won't load filtered-out rows
- **Combines multiple operations** into fewer passes over the data
- Saves time and memory on large datasets

---
## Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Week6Assignment").getOrCreate()

df = spark.read.csv(
    "data/source.csv",
    header=True,        # First row is treated as column names
    inferSchema=True    # Spark automatically detects data types
)

df.show(5)
df.printSchema()

---
## Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

| Feature | CSV | Parquet |
|---|---|---|
| Storage format | Row-based | Columnar (column-by-column) |
| Schema stored? | No | Yes (schema embedded) |
| Compression | Poor | Excellent |
| Read speed | Slow (reads entire row) | Fast (reads only needed columns) |
| Human readable | Yes | No (binary format) |

**Why it matters for performance:**
- If you only need 3 columns out of 100, **Parquet reads only those 3 columns** → much less I/O
- CSV reads the **entire row** even if you don't need most of it → wasteful
- Parquet uses better compression → **smaller file size**, faster transfer
- Parquet supports **Predicate Pushdown** → skips entire row groups

---
## Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.

In [ ]:
# Method 1: Using DataFrame API
result = df.select("product_id", "price") \
           .filter(df["category"] == "Electronics")

result.show()

# Method 2: Using SQL-style filter
result2 = df.select("product_id", "price") \
            .where("category = 'Electronics'")

result2.show()

---
## Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from String to Double.

In [ ]:
from pyspark.sql.functions import col

df_revised = df.withColumnRenamed("old_name", "new_name") \
               .withColumn("price", col("price").cast("Double"))

df_revised.printSchema()
df_revised.show(5)

---
## Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

**What is a Lineage Graph (DAG)?**
- DAG = Directed Acyclic Graph
- Spark records every transformation applied to data as a **lineage**
- It knows exactly how each RDD/DataFrame was created from the source

**How fault tolerance works:**
- If a worker node **fails mid-job**, Spark doesn't crash
- It looks at the **DAG lineage** to find out which data was on that node
- It **re-computes only the lost partitions** from the original source using the stored info

**Example:**
```
Source CSV → filter → groupBy → count
```
If the `groupBy` step fails, Spark re-runs from `filter` step using the lineage — not from scratch.

---
## Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000.

In [ ]:
# Filter with AND condition
df_filtered = df_orders.filter(
    (df_orders["status"] == "Completed") & (df_orders["amount"] > 1000)
)

df_filtered.show()

# Alternative using SQL string syntax
df_filtered2 = df_orders.filter("status = 'Completed' AND amount > 1000")

df_filtered2.show()

---
## Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

**What is Predicate Pushdown?**
- A "predicate" is a filter condition (e.g., `age > 30`)
- "Pushdown" means the filter is applied **at the file/storage level**, before data is loaded into memory

**How it works with Parquet:**
- Parquet stores **min/max statistics** for each column in every "row group"
- When you apply a filter, Spark checks these stats first
- If a row group **can't possibly contain** matching rows, Spark **skips it entirely**
- Only matching row groups are loaded into memory

**Effect on performance:**
- Much **less data read from disk** → less I/O
- Much **less data in memory** → less RAM usage
- **Faster query execution** overall

**Example:** If filtering `year = 2024` and a row group has min=2020, max=2022, Spark skips that entire row group without reading it.

---
## Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

In [ ]:
from pyspark.sql.functions import col

df_with_tax = df.withColumn("final_price", col("base_price") * 1.18)

df_with_tax.select("base_price", "final_price").show()

---
## Q11: What is the difference between Transformations and Actions? Provide two examples of each.

| | Transformations | Actions |
|---|---|---|
| What they do | Define a new operation on data | Trigger actual execution |
| Lazy? | Yes (not executed immediately) | No (executes the whole plan) |
| Return type | New DataFrame/RDD | A result (number, list, file) |

**Transformation Examples:**
1. `.filter()` — filters rows based on a condition
2. `.select()` — selects specific columns

**Action Examples:**
1. `.count()` — returns the number of rows (triggers execution)
2. `.show()` — prints rows to screen (triggers execution)

In [ ]:
# Transformations (lazy - nothing runs yet)
df_transformed = df.filter(col("age") > 18)
df_selected = df_transformed.select("name")

# Actions (these trigger actual execution)
print(df_selected.count())
df_selected.show(5)

---
## Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".

In [ ]:
from pyspark.sql.functions import col

# Step 1: Load Parquet file
df_parquet = spark.read.parquet("path/to/input")

# Step 2: Filter out rows where user_id is null
df_clean = df_parquet.filter(col("user_id").isNotNull())

# Step 3: Save as CSV
df_clean.write.csv(
    "path/to/output",
    header=True,       # Include column names in CSV
    mode="overwrite"   # Overwrite if folder already exists
)

---
## Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

| | Client Mode | Cluster Mode |
|---|---|---|
| Where Driver runs | On the **client machine** (your laptop/local machine) | Inside the **cluster** (on a worker/master node) |
| Network dependency | Driver must stay connected to cluster the whole time | Driver runs independently inside cluster |
| If client disconnects | Job **fails** | Job **continues** |
| Best for | Interactive use, debugging, notebooks | Production jobs, long-running pipelines |

- **Client Mode** → good for development (you can see logs directly)
- **Cluster Mode** → good for production (job doesn't depend on your machine staying on)

---
## Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

In [ ]:
df_result = df.filter(
    (df["region"] == "North") | (df["priority"] == "High")
)
df_result.show()

# Alternative using SQL string syntax
df_result2 = df.filter("region = 'North' OR priority = 'High'")
df_result2.show()

---
## Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?

**`.show(5)`:**
- Fetches and displays only **5 rows**
- Data stays on the cluster, only a tiny preview is sent to the Driver
- Safe to use on any size dataset
- Fast and memory-efficient

**`.collect()`:**
- Pulls **ALL rows** from the entire cluster into the Driver's memory
- On a multi-terabyte dataset → Driver runs **out of memory** and crashes
- Very slow (huge data transfer over network)
- Can kill your entire Spark application